# Notebook 22f — Wu 2003 S-B3: held-out SBC confirmation set (widened lognormal prior)

## Purpose

Generates ONE dedicated, adequately-powered SBC confirmation set (N=800) for the 2026-09-25
large-scale S-B3 campaign, drawn from the exact same widened prior used for every training chunk
(`mixed_wide_prior_4d_sb3(uniform_idx=(0,1), sigma=LOGN_SIGMA_SB3_4D_WIDE)`). Per
`finding_sbc_power_eta_col` (N=200 SBC gives false passes; N>=400 is this project's floor), N=800
is used here for a stronger final claim.

**Run this ONCE, centrally — not on every generation machine.** It uses a single reserved seed
(`SEED=999999`) that must never be assigned to a training-chunk machine, and its output is **never
pooled into training** (`notebooks/22g` only globs `..._seed*_n*.npz` training banks, which this
notebook does not produce under that pattern) — this is what keeps the eventual SBC check in
`notebooks/24d` an honest, held-out calibration claim rather than a self-consistency check on the
training data itself. Same rationale as `notebooks/48`'s pooling registry, which deliberately
excludes its own confirm sets from pooling.

**Not executed** — a run of this scope takes hours; launch under `caffeinate`/`nohup`.

## Output file

- `results/22f_sb3_widelognormal_sbc_confirm_n800.npz`


In [ ]:
import pathlib, sys, time, importlib
import numpy as np

cwd = pathlib.Path.cwd().resolve()
ROOT = cwd if (cwd / 'src').exists() else cwd.parent
sys.path.insert(0, str(ROOT / 'src'))

import cstr_sbi.recycle.sb3 as sb3
sb3 = importlib.reload(sb3)
from cstr_sbi.recycle.sb3 import (
    LOGN_SIGMA_SB3_4D_WIDE,
    mixed_wide_prior_4d_sb3,
    deterministic_window_sb3,
    noisy_replicates_sb3,
    nominal_warm_start_sb3,
)
from cstr_sbi.recycle.summaries import N_SUMMARIES_SB, compute_summaries

DATA = ROOT / 'data'; DATA.mkdir(exist_ok=True)
RESULTS = ROOT / 'results'; RESULTS.mkdir(exist_ok=True)
LOGS = ROOT / 'logs'; LOGS.mkdir(exist_ok=True)

# Reserved -- never assign this seed to a training-chunk machine (notebooks/22e).
CONFIRM_SEED = 999999
N_CONFIRM = 800

NOISE_PCT = 0.003
T_FINAL_H = 4.0
N_SAVE = 240
CHECKPOINT_EVERY = 40
TIME_CHECKPOINT_S = 60
PROGRESS_EVERY_ATTEMPTS = 10
STEADY_MAX_REL_DRIFT = 0.05
MAX_NORM_CHANNEL = 3.0
MAX_ABS_RAW_OBS = 3.0e7
PRIOR_TAG = 'widelognormal_v1'

PROGRESS_LOG_PATH = LOGS / 'notebook22f_progress.log'
CONFIRM_PATH = RESULTS / f'22f_sb3_widelognormal_sbc_confirm_n{N_CONFIRM}.npz'

import torch
torch.manual_seed(CONFIRM_SEED)
rng = np.random.default_rng(CONFIRM_SEED)
prior = mixed_wide_prior_4d_sb3(uniform_idx=(0, 1), sigma=LOGN_SIGMA_SB3_4D_WIDE)
y0_sb3 = nominal_warm_start_sb3()

print(f'ROOT={ROOT}')
print(f'CONFIRM_SEED={CONFIRM_SEED} (reserved -- must never match a notebooks/22e SB3_GEN_SEED)')
print(f'N_CONFIRM={N_CONFIRM}')
print(f'CONFIRM_PATH={CONFIRM_PATH.relative_to(ROOT)}')


## Section 1 — Generation/diagnostics/checkpointing helpers (identical logic to notebooks/22e, self-contained so this notebook stands alone)


In [ ]:
def simulate_summary_sb3(theta4, sim_rng, noise_pct=NOISE_PCT, t_final_h=T_FINAL_H, n_save=N_SAVE):
    theta4 = np.asarray(theta4, dtype=np.float32)
    t_h, raw_obs = deterministic_window_sb3(theta4, y0=y0_sb3, t_final_h=t_final_h, n_save=n_save)
    raw_obs = np.asarray(raw_obs)
    noisy = noisy_replicates_sb3(raw_obs, n_replicates=1, rng=sim_rng, noise_pct=noise_pct)[0]
    summary = compute_summaries(noisy, 'S-B', np.asarray(t_h))
    return np.asarray(t_h), raw_obs, noisy, summary


def window_diag(raw_obs):
    raw_obs = np.asarray(raw_obs)
    check_idx = [0, 1, 2, 5, 6, 7, 8, 9, 10]
    norm_idx = [7, 8, 9, 10]
    q_start = max(0, 3 * len(raw_obs) // 4)
    last_q = raw_obs[q_start:]
    final = raw_obs[-1]
    q_mean = last_q.mean(axis=0)
    denom = np.maximum(np.abs(q_mean), 1e-6)
    rel_drift = np.abs(final - q_mean) / denom
    return {
        'max_rel_drift_sb': float(np.max(rel_drift[check_idx])),
        'max_norm_channel': float(np.max(np.abs(raw_obs[:, norm_idx]))),
        'max_abs_raw': float(np.max(np.abs(raw_obs))),
    }


def is_acceptable_window(diag):
    return (
        diag['max_rel_drift_sb'] <= STEADY_MAX_REL_DRIFT
        and diag['max_norm_channel'] <= MAX_NORM_CHANNEL
        and diag['max_abs_raw'] <= MAX_ABS_RAW_OBS
    )


def load_or_init_bank(out_path):
    for candidate in (out_path, out_path.with_suffix('.partial.npz')):
        if candidate.exists():
            data = np.load(candidate)
            theta_rows = list(np.asarray(data['thetas4'], dtype=np.float32))
            summary_rows = list(np.asarray(data['summaries'], dtype=np.float32))
            attempts = int(data['attempts']) if 'attempts' in data.files else len(theta_rows)
            return theta_rows, summary_rows, attempts
    return [], [], 0


def save_bank(out_path, theta_rows, summary_rows, attempts):
    theta_arr = np.asarray(theta_rows, dtype=np.float32)
    summary_arr = np.asarray(summary_rows, dtype=np.float32)
    assert summary_arr.shape[1] == N_SUMMARIES_SB
    np.savez(
        out_path,
        thetas4=theta_arr,
        summaries=summary_arr,
        seed=CONFIRM_SEED,
        attempts=attempts,
        structure='S-B3',
        prior_tag=PRIOR_TAG,
        role='sbc_confirm_holdout',
        sigma_eta_col=float(LOGN_SIGMA_SB3_4D_WIDE[2]),
        sigma_xi_reb=float(LOGN_SIGMA_SB3_4D_WIDE[3]),
        t_final_h=T_FINAL_H,
        n_save=N_SAVE,
    )


def _emit_progress(msg):
    line = f'[{time.strftime("%H:%M:%S")}] {msg}'
    print(line)
    with open(PROGRESS_LOG_PATH, 'a') as f:
        f.write(line + '\\n')


def generate_bank(out_path, n_target, sim_rng, checkpoint_every=CHECKPOINT_EVERY,
                   time_checkpoint_s=TIME_CHECKPOINT_S,
                   progress_every_attempts=PROGRESS_EVERY_ATTEMPTS):
    theta_rows, summary_rows, attempts = load_or_init_bank(out_path)
    if len(theta_rows) >= n_target:
        _emit_progress(f'{out_path.name}: already has {len(theta_rows)}/{n_target} rows, done.')
        return (np.asarray(theta_rows, dtype=np.float32)[:n_target],
                np.asarray(summary_rows, dtype=np.float32)[:n_target])

    _emit_progress(f'{out_path.name}: resuming with {len(theta_rows)}/{n_target} accepted rows, '
                    f'{attempts} attempts so far.')
    partial_path = out_path.with_suffix('.partial.npz')
    t_start = time.time()
    t_last_checkpoint = t_start
    reject_nonfinite = 0
    reject_diag = 0

    while len(theta_rows) < n_target:
        theta4 = prior.sample((1,)).numpy()[0]
        attempts += 1
        try:
            _, raw_obs, _, summary = simulate_summary_sb3(theta4, sim_rng)
        except Exception as exc:
            reject_nonfinite += 1
            if attempts % progress_every_attempts == 0:
                _emit_progress(f'attempt {attempts}: solver exception ({exc!r}), rejected.')
            continue

        if not np.isfinite(summary).all():
            reject_nonfinite += 1
            continue
        diag = window_diag(raw_obs)
        if not is_acceptable_window(diag):
            reject_diag += 1
            continue

        theta_rows.append(theta4.astype(np.float32))
        summary_rows.append(summary.astype(np.float32))

        if attempts % progress_every_attempts == 0:
            elapsed = time.time() - t_start
            rate = len(theta_rows) / max(elapsed, 1e-6)
            _emit_progress(
                f'attempt {attempts}: accepted={len(theta_rows)}/{n_target} '
                f'(reject_nonfinite={reject_nonfinite}, reject_diag={reject_diag}), '
                f'{rate:.3f} accepted/s'
            )

        now = time.time()
        if len(theta_rows) % checkpoint_every == 0 or (now - t_last_checkpoint) > time_checkpoint_s:
            save_bank(partial_path, theta_rows, summary_rows, attempts)
            t_last_checkpoint = now
            _emit_progress(f'checkpointed {len(theta_rows)}/{n_target} rows to {partial_path.name}')

    save_bank(out_path, theta_rows, summary_rows, attempts)
    if partial_path.exists():
        partial_path.unlink()
    accept_rate = len(theta_rows) / max(attempts, 1)
    _emit_progress(
        f'DONE {out_path.name}: {len(theta_rows)} rows, {attempts} attempts, '
        f'accept_rate={accept_rate:.3f}, reject_nonfinite={reject_nonfinite}, reject_diag={reject_diag}'
    )
    return np.asarray(theta_rows, dtype=np.float32), np.asarray(summary_rows, dtype=np.float32)


## Section 2 — Generate the held-out confirm set


In [ ]:
theta_confirm, summary_confirm = generate_bank(CONFIRM_PATH, N_CONFIRM, rng)
print(f'Held-out SBC confirm set: {theta_confirm.shape}, {summary_confirm.shape}')
print(f'Saved to {CONFIRM_PATH.relative_to(ROOT)} -- do NOT add this file to notebooks/22g\'s '
      'pooling glob pattern.')
